# B2-024-gpu-scientific-ml-capstone — Practice p23 — Solution

**Type:** scenario · **Difficulty:** core · **Concepts:** gpu-colab-l4-workflow

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Solution

All arithmetic is reproduced in the code cell below. Units: 1 GB $=10^9$ bytes.

### (a) Memory

Session 1's model: $16P$ bytes of `float32` state (weights, gradients, Adam $m$ and $v$) plus activations $Be\times$(bytes per element).

* $16P=16\cdot48{,}000{,}000=768{,}000{,}000$ B $=0.768$ GB.
* Activation elements at $B=64$: $64\cdot1{,}500{,}000=96{,}000{,}000$.
* `bfloat16` activations: $2\cdot96\times10^6=192{,}000{,}000$ B; total $960{,}000{,}000$ B $=0.96$ GB.
* all-`float32`: $4\cdot96\times10^6=384{,}000{,}000$ B; total $1{,}152{,}000{,}000$ B $=1.152$ GB.
* Budget: $0.7\cdot24\times10^9=16.8\times10^9$ B. Largest batch with `bfloat16` activations:
$B_{\max}=\left\lfloor\dfrac{16.8\times10^9-0.768\times10^9}{2\cdot1.5\times10^6}\right\rfloor=\lfloor5344\rfloor=5344$.

At batch 64 the plan uses under 6% of the 16.8 GB budget, so **time, not memory, is the binding constraint** (the throughput measurement and the target step count are both at batch 64).

### (b) Time budget

Session 1's split of the 180 minutes:

| part | share | minutes |
|---|---:|---:|
| setup and smoke test | 10% | 18 |
| main run | 60% | 108 |
| evaluation and writeup | 20% | 36 |
| restart slack | 10% | 18 |

Main run: $60{,}000/9.5=6{,}315.8$ s $=105.3$ min of training, plus the checkpoint writes from (c), $24\cdot6=144$ s $=2.4$ min, for $107.7$ min. This **fits** the 108-minute share, but with only about 0.3 min of margin, so I keep the 60,000-step target and batch 64 and protect it in two ways: a disconnect is paid for from the 18-minute restart slack (at most 5 minutes of lost training plus about 3 minutes to reconnect, re-run setup and load), and a stop rule: if the measured rate over the first 2,500 steps is below 9.5 steps/s, I lower the step target to $\lfloor\text{rate}\cdot(108\cdot60-144)\rfloor$ before continuing and record the change in the log rather than eating into evaluation time. I do not raise the batch size, since the 9.5 steps/s figure was measured only at batch 64 and changing it changes the experiment.

### (c) Checkpoint cadence

Losing at most 5 minutes means at most $300\cdot9.5=2{,}850$ steps between checkpoints. I choose **every 2,500 steps** ($2500/9.5=263$ s $\approx4.4$ min of training), which divides 60,000 evenly: **24 checkpoints** (steps 2,500, 5,000, …, 60,000), total write time $24\cdot6=144$ s $=2.4$ min.
File size: $12P=12\cdot48\times10^6=576{,}000{,}000$ B $=0.576$ GB each.
I keep `latest.pt` (overwritten through an atomic rename of `latest.pt.tmp`, so a disconnect during a write never corrupts it) plus `best.pt`, the checkpoint with the best validation score so far; that is at most about 1.15 GB on persistent storage, and it never fills the drive.

### (d) OOM triage

*Failure 1 — batch 256, OOM at step 1.* By (a), batch 256 should need only $0.768+2\cdot256\cdot1.5\times10^6/10^9=1.536$ GB, so an immediate OOM means the estimate is not what is running. Order: (1) check what else holds GPU memory: an earlier cell's model, optimizer or batch still referenced, or a previous failed run's tensors kept alive by a traceback, so restart the runtime and re-run setup only; (2) verify the run is what I think: `bfloat16` autocast really on, the batch really 256 and not the whole dataset, evaluation under `torch.no_grad()`; (3) measure: one step at batch 64 with `torch.cuda.reset_peak_memory_stats()` and `torch.cuda.max_memory_allocated()`, and compare with the 0.96 GB estimate to find which term (e.g. $e$) was underestimated; (4) only then lower the batch, using gradient accumulation if the recipe needs the larger effective batch; (5) `torch.cuda.empty_cache()` last, since it cannot free referenced tensors.

*Failure 2 — batch 64, OOM at step 3,000 after a steady rise.* A steady per-step rise at a fixed batch size is a **leak**, not a sizing problem, and the most likely cause is storing a tensor that keeps its graph alive, typically `losses.append(loss)` instead of `losses.append(loss.item())` (or accumulating `running_loss += loss`, or keeping logits/predictions from a periodic evaluation run without `torch.no_grad()`). Order: (1) inspect every list or accumulator in the loop and the evaluation code for stored tensors and fix with `.item()`/`.detach()` and `torch.no_grad()`; (2) confirm the fix by logging `torch.cuda.max_memory_allocated()` every 100 steps: it must be flat; (3) resume from the last checkpoint (step 2,500) with the fix, since the leak did not change the weights; shrinking the batch would only postpone the crash.

### (e) Restart procedure

1. Reconnect to an L4 runtime and confirm `torch.cuda.is_available()` and the device name.
2. Re-run the setup cell: imports, the data module, seeds, and the **same** config dict (printed and compared with the copy in the log).
3. Mount persistent storage and locate `latest.pt` (ignore any leftover `latest.pt.tmp`).
4. Build a fresh model and optimizer from the config, move the model to the GPU, then `load_checkpoint`: model and optimizer state dicts, step, and the CPU and CUDA RNG states.
5. Verify continuity: the restored step equals the last checkpoint step in the experiment log; one validation pass reproduces the logged validation score at that step (to a small tolerance on the GPU); and the config hash matches.
6. Record the restart (time, step, lost steps) in the log, then continue the loop from `step + 1` with the same checkpoint cadence.

### (f) Submission checklist

1. The notebook runs top to bottom on the CPU config (`device="cpu"`, `amp_dtype=None`) on a machine without a GPU.
2. Every GPU-only cell is guarded by `torch.cuda.is_available()`, and the L4 config differs from the CPU config only in the dictionary.
3. Seeds and both config dicts are printed in the notebook.
4. All selection (checkpoint choice, hyperparameters, number of steps) used validation data only.
5. The locked test set was scored exactly once, after all fitting and selection, and the call count is asserted.
6. The test score is reported with its bootstrap interval next to the committed baseline under the same protocol.
7. Required outputs (predictions, writeup, log including restarts) are saved where the grader expects them.
8. No downloaded weights, external data, or checkpoints from other runs are referenced; checkpoints used are from this run.

## Arithmetic check

In [ ]:
import math

P = 48_000_000
e = 1_500_000
B = 64
state = 16 * P
mem_bf16 = state + 2 * B * e
mem_fp32 = state + 4 * B * e
budget_bytes = 7 * 24 * 10**9 // 10          # 70% of 24 GB, in exact integer bytes
B_max = (budget_bytes - state) // (2 * e)

split = {"setup": 18, "main": 108, "eval": 36, "slack": 18}
train_minutes = 60_000 / 9.5 / 60
interval = 2500
max_interval = 5 * 60 * 9.5
n_ckpt = 60_000 // interval
ckpt_minutes = n_ckpt * 6 / 60
ckpt_gb = 12 * P / 1e9
main_needed = train_minutes + ckpt_minutes
mem_256 = state + 2 * 256 * e
print(f"bf16 {mem_bf16 / 1e9:.3f} GB, fp32 {mem_fp32 / 1e9:.3f} GB, B_max {B_max}, batch-256 estimate {mem_256 / 1e9:.3f} GB")
print(f"training {train_minutes:.2f} min + checkpoints {ckpt_minutes:.2f} min = {main_needed:.2f} min of {split['main']} min")
print(f"interval {interval} steps ({interval / 9.5 / 60:.2f} min), {n_ckpt} checkpoints, {ckpt_gb:.3f} GB each")

### Answer check

In [ ]:
assert state == 768_000_000 and mem_bf16 == 960_000_000 and mem_fp32 == 1_152_000_000
assert B_max == 5344
assert sum(split.values()) == 180
assert main_needed <= split["main"]
assert interval <= max_interval and 60_000 % interval == 0 and n_ckpt == 24
assert abs(ckpt_minutes - 2.4) < 1e-12 and abs(ckpt_gb - 0.576) < 1e-12
assert mem_256 == 1_536_000_000 and mem_256 < budget_bytes